# 01. Toolkit Playground

**Purpose:** Welcome notebook. Learn how to run the toolkit safely, make a first API call on a single address, and explore how generation parameters (especially `temperature`) affect output.

**Key takeaway:** You can control LLM behavior; there is a trade‑off between consistency and creativity.

## Step 1: Set up environment & authenticate

In [ ]:
!git clone https://github.com/alxefremov/esmt-workshop.git
%pip install -q -r esmt-workshop/requirements.txt

import os
from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report
from esmt_workshop.experiment_logging import load_experiment_history, log_experiment_run
from esmt_workshop.student_api import process_batch_addresses, process_single_address
from esmt_workshop.load_utils import get_roots
from esmt_workshop.authenticate import authenticate

roots = get_roots()
PROJECT_ROOT = roots["PROJECT_ROOT"]
SRC_DIR = roots["SRC_DIR"]
STUDENT_EMAIL = authenticate()


## TO BE REMOVED

In [ ]:
# Safe mode for workshops: set True to run without calling the gateway.
MOCK_MODE = True

print('STUDENT_EMAIL =', STUDENT_EMAIL)
print('MOCK_MODE =', MOCK_MODE)


## Step 4: Your First Call (Single Address)

Run one address through the parser to see the input → output flow.

In [ ]:
# Use the baseline stage for first experiments.
STAGE_NAME = 'baseline'
MODEL_NAME = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')

# Prompt is editable directly in the notebook.
USE_CUSTOM_PROMPT = True
PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
1. Town Name is only city/town/locality.
2. Postal Code includes only postal token(s).
3. Remaining Address includes everything else.
4. Country Code is ISO alpha-2 uppercase.
5. Use empty strings when uncertain.
'''

custom_prompt = PROMPT_TEMPLATE if USE_CUSTOM_PROMPT else None

sample_address = '332 Menzies Street, Victoria, BC V8V 2G9, Canada'
print('Sample address:', sample_address)

result = process_single_address(
    sample_address,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=MODEL_NAME,
    temperature=0.0,
    top_p=0.95,
    top_k=40,
    max_tokens=None,
    use_guardrails=False,
    custom_prompt_template=custom_prompt,
    kb_csv_path=str(PROJECT_ROOT / 'data/input/address_formats.csv'),
    mock_mode=MOCK_MODE,
)

result


## Step 5: Parameter Playground (Temperature)

Try different `temperature` values on the same address and compare outputs.

- `temperature = 0.0`: most consistent
- `temperature > 0.0`: more variation

In [ ]:
play_address = 'Stoke-on-Trent, United Kingdom'
temps = [0.0, 0.2, 0.7]

rows = []
for t in temps:
    out = process_single_address(
        play_address,
        email=STUDENT_EMAIL,
        stage=STAGE_NAME,
        model=MODEL_NAME,
        temperature=t,
        top_p=0.95,
        top_k=40,
        max_tokens=None,
        use_guardrails=False,
        custom_prompt_template=custom_prompt,
        kb_csv_path=str(PROJECT_ROOT / 'data/input/address_formats.csv'),
        mock_mode=MOCK_MODE,
    )
    out = dict(out) if isinstance(out, dict) else out
    out['temperature'] = t
    rows.append(out)

pd.DataFrame(rows)


## Step 6: What to Try Next

- Flip `MOCK_MODE = False` (if instructed by organizers) and re-run Step 5.
- Modify the prompt rules and observe how the output changes.
- Try different addresses (with/without postal code, different countries).
